# The plan, step by step

This notebook has one cell for each step of [PLAN.md](https://github.com/gbirhanu/ondevice-hallu/blob/main/PLAN.md)
that needs the GPU. The step numbers are the same as in the plan.

**How to use it:** choose a T4 GPU (Runtime > Change runtime type; step 13 needs only a CPU), run the two setup
cells, then run only the cell of the step you are doing. Results go to `MyDrive/ondevice_hallu/work`, next to the earlier results.
Every command skips work that is already saved, so after a disconnect, run the two setup cells and the same
step cell again: it continues where it stopped. Keep the browser tab open while a step runs.

| Step | What it does | Needs | Time on a free T4 |
|---|---|---|---|
| 5 | Test A at 68M: small-model answers against 7B answers | nothing | about 2.5 hours |
| 6 | Two more training runs (seeds 2 and 3) of the four main students | step 2 agreed | 1.2 + 5.5 hours |
| 7 | Two more training runs of test A at 17M | step 2 agreed | about 1 hour |
| — | The averaged tables (`results/summary.md`) | any of 5–7 | 15–30 minutes |
| 12 | Baseline: phone-sized chat models (0.5B, 1.5B) as the checker | nothing | about 30 minutes |
| 13 | The 8-bit files of our 68M and 17M checkers: size and accuracy | step 8 | about 2 hours, **CPU only** |
| 9–11, 13, 17, 20 | Added here when we reach them | see PLAN.md | |

Everything is free: only the Colab GPU is used, no paid service.

In [ ]:
# Setup 1 of 2: Drive, code, data and libraries (about 3 minutes)
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, subprocess, shutil
from pathlib import Path
from IPython.display import Markdown, display

REPO = 'https://github.com/gbirhanu/ondevice-hallu.git'
CODE = Path('/content/ondevice_hallu')
WORK = Path('/content/drive/MyDrive/ondevice_hallu/work')
DATA = Path('/content/RAGTruth')

def sh(command):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    print(result.stdout[-3000:], flush=True)
    return result.returncode

if (CODE / '.git').exists():
    sh(['git', '-C', str(CODE), 'pull', '--ff-only'])
else:
    assert sh(['git', 'clone', REPO, str(CODE)]) == 0, f'Cannot clone {REPO}.'
if not DATA.exists():
    sh(['git', 'clone', '--depth', '1', 'https://github.com/ParticleMedia/RAGTruth.git', str(DATA)])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'accelerate', 'bitsandbytes', 'datasets',
    'onnx', 'onnxruntime==1.29.0', 'onnx-ir==1.0.0', 'psutil'])
try:                                     # optional: a Hugging Face token in the Secrets panel (key icon)
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass                                 # not needed for these steps
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU. That is fine for step 13 (CPU only); every other step needs Runtime > Change runtime type > T4 GPU.')

In [ ]:
# Setup 2 of 2: helpers used by every step
def run(*arguments):
    """Runs one run.py command and shows its output as it comes. Returns True when it worked."""
    command = [sys.executable, '-u', str(CODE / 'run.py'), *map(str, arguments),
               '--data-dir', str(DATA), '--work-dir', str(WORK)]
    process = subprocess.Popen(command, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               env={**os.environ, 'TOKENIZERS_PARALLELISM': 'false'})
    for line in process.stdout:
        if not any(x in line for x in ('Loading checkpoint shards', 'Warning', 'Writing model shards', 'Loading weights')):
            print(line, end='', flush=True)
    return process.wait() == 0

def free_disk():
    """Deletes downloaded models from Colab's own disk (never from Drive)."""
    hub = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface')) / 'hub'
    for d in hub.glob('models--*'):
        shutil.rmtree(d, ignore_errors=True)

def show(name):
    display(Markdown((WORK / 'results' / name).read_text()))

def kept(gens):
    """New answers kept per answering model (written by "generate")."""
    return {g: json.loads((WORK / 'results' / f'extra_{g}.json').read_text())['kept']
            for g in gens if (WORK / 'results' / f'extra_{g}.json').exists()}

TEACHER = 'ld_large'                                       # LettuceDetect-large, 396M
SMALL = ['qwen2.5-0.5b', 'qwen2.5-1.5b', 'smollm2-1.7b']   # phone-sized answering models
LARGE = ['qwen2.5-7b', 'olmo2-7b', 'mistral-7b-v0.3']      # 7B answering models (OLMo-2 did not run)
small, large = kept(SMALL), kept(LARGE)
N = min(sum(small.values()), sum(large.values())) if small and large else 0
SMALL_RUN = 'kd{size}' if sum(small.values()) <= N * 1.02 else 'kd{size}-n%d' % N   # same number of new answers

def train_and_score(name, size, *options, epochs=5):
    """Trains a student exactly like kd17m / kd68m (Ettin, 5 epochs, learning rate 2e-5), then scores it."""
    ok = run('train', '--name', name, '--base', f'jhu-clsp/ettin-encoder-{size}', '--epochs', epochs,
             '--lr', 2e-5, '--teacher', TEACHER, *options)
    return ok and run('score', '--name', name)

# The four main students as trained on 18 Oct: (size, epochs, options). Step 6 repeats them with other seeds.
MAIN = {'hard17m-8ep': ('17m', 8, []), 'kd17m': ('17m', 5, ['--teacher', TEACHER, '--extra', *small]),
        'hard68m-8ep': ('68m', 8, []), 'kd68m': ('68m', 5, ['--teacher', TEACHER, '--extra', *small])}

def again(base_run, seed):
    """Trains one of MAIN again with another seed (the run is named <run>-s<seed>), then scores it."""
    size, epochs, options = MAIN[base_run]
    name = f'{base_run}-s{seed}'
    ok = run('train', '--name', name, '--base', f'jhu-clsp/ettin-encoder-{size}', '--epochs', epochs,
             '--lr', 2e-5, '--seed', seed, *options)
    assert ok and run('score', '--name', name), f'{name} failed; see the message above.'

print('new answers kept, small models:', small)
print('new answers kept, 7B models:   ', large, '-> each test A student gets', N, 'new answers')
need = ['results/kd68m_training.json', 'results/kd17m-n4509_score.json', 'results/kdlarge17m_score.json',
        'preds/ld_large_train.json.gz', 'preds/ld_large_test.json.gz', 'preds/psilo_gold.json.gz',
        *[f'preds/{TEACHER}_extra_{g}.json.gz' for g in [*small, *large]]]
missing = [n for n in need if not (WORK / n).exists()]
print('Earlier results this notebook needs, missing:', missing or 'none (good)')

## Step 5 — Test A at 68M

Does it matter **who writes** the new answers the student learns from? At 17M (19 Oct), the student trained on
4,509 answers from the small models (0.5–1.7B) beat the one trained on 4,509 answers from 7B models by 2.5
points on PsiloQA. This step trains the same pair at 68M:

- `kd68m-n4509`: 4,509 new answers from the small models;
- `kdlarge68m`: 4,509 new answers from Qwen2.5-7B and Mistral-7B v0.3.

Everything else is the same as `kd68m`. Each run takes about 70 minutes; then both are scored on RAGTruth and
PsiloQA, and the table is written to `results/compare_step5.md` (the older `compare.md` files are not touched).
The two new models take about 0.3 GB on Drive.

In [ ]:
# Step 5: test A at 68M (about 2.5 hours)
size = '68m'
small_run = SMALL_RUN.format(size=size)                                  # kd68m-n4509
assert train_and_score(small_run, size, '--extra', *small, '--extra-max', N), 'Training failed; see the message above.'
assert train_and_score(f'kdlarge{size}', size, '--extra', *large, '--extra-max', N), 'Training failed; see the message above.'
if not run('psilo'):                                                     # scores only runs not scored yet
    print('--> PsiloQA scoring failed (see above); the table below then has RAGTruth columns only.')
pairs = [f'{SMALL_RUN.format(size="17m")}:kdlarge17m',                    # test A at 17M (19 Oct), for reference
         f'{small_run}:kdlarge{size}',                                    # test A at 68M: the new result
         f'kd{size}:{small_run}']                                         # what 4,509 instead of 6,719 new answers cost
run('compare', '--pairs', *pairs, '--out', 'compare_step5.md')
show('compare_step5.md')

**How to read it.** Each cell is *span F1 of A → span F1 of B (B minus A, 95% interval)*.

- **Row 2 (`kd68m-n4509 → kdlarge68m`) is the new result.** A negative number means the student that learned
  from **small** models' answers is better. At 17M (row 1) the PsiloQA difference was −2.50 [−3.17, −1.79] and
  the RAGTruth difference −1.02 [−2.69, 0.55] (no clear difference). If row 2 points the same way, the finding
  holds at both sizes. It is still one training run per student; steps 6 and 7 add two more seeds.
- **Row 3** shows how much the 68M student loses with 4,509 instead of 6,719 new answers.
- When it is done, tell Claude "step 5 done": Claude reads the table from Drive and updates the status log.

## Step 6 — Two more training runs of the four main students

**Run this only after `EVALUATION.md` (step 2) is agreed and pushed.** The four students of 18 October are
trained again with seeds 2 and 3, with exactly the same recipe; `run.py` refuses to start a run whose recipe
differs from the first run's. A run named `kd17m-s2` is `kd17m` trained with seed 2.

| Student | What it learns from | One run on a T4 |
|---|---|---|
| `hard17m-8ep`, `hard68m-8ep` | people's labels only, 8 epochs | 17 / 83 minutes |
| `kd17m`, `kd68m` | people's labels + the teacher, and 6,719 new answers from small models | 16 / 77 minutes |

The work is split into two cells so that each fits in one Colab session: the 17M runs (about 1.2 hours) and the
68M runs (about 5.5 hours; if the session ends, run the setup cells and the same cell again: finished runs are
skipped in seconds). The 8 new models take about 0.7 GB on Drive.

In [ ]:
# Step 6, 17M part: 4 runs, about 1.2 hours
for seed in (2, 3):
    for base_run in ('hard17m-8ep', 'kd17m'):
        again(base_run, seed)

In [ ]:
# Step 6, 68M part: 4 runs, about 5.5 hours (it can be spread over two sessions)
for seed in (2, 3):
    for base_run in ('hard68m-8ep', 'kd68m'):
        again(base_run, seed)

## Step 7 — Two more training runs of test A at 17M

**Also only after step 2.** `kd17m-n4509` (4,509 new answers from small models) and `kdlarge17m` (4,509 from
7B models), each with seeds 2 and 3: 4 runs, about 1 hour. It needs only the two setup cells.

In [ ]:
# Step 7: test A at 17M with seeds 2 and 3 (about 1 hour)
for seed in (2, 3):
    for name, options in [(SMALL_RUN.format(size='17m'), ['--extra', *small, '--extra-max', N]),
                          ('kdlarge17m', ['--extra', *large, '--extra-max', N])]:
        assert train_and_score(f'{name}-s{seed}', '17m', *options, '--seed', seed), 'Training failed; see above.'

## The averaged tables (after steps 5, 6 and 7)

Scores the new runs on PsiloQA and writes `results/summary.md`: every checker as the mean ± spread of its
training runs, and the differences the success criteria need, run by run, with "Counts?" applying rule R3 (all
three runs point the same way and the 95% interval excludes 0). It can be run at any time; runs that are missing
are left out. It takes 15–30 minutes, mostly scoring the new runs on PsiloQA (2–3 minutes each). Then tell Claude
"summary done" (step 8).

In [ ]:
# The averaged tables: results/summary.md and results/summary.json
if not run('psilo'):
    print('--> PsiloQA scoring failed (see above); the summary then has fewer PsiloQA numbers.')
assert run('summary'), 'The summary failed; see the message above.'
show('summary.md')

## Step 12 — Baseline: can the phone's own chat model check answers?

A phone that runs a 0.5B or 1.5B chat model could ask **that same model** to list the unsupported parts of an
answer, with no extra checker. This step measures how well that works, on the same 300 RAGTruth test answers on
which Qwen2.5-7B scored 11.1 (19 Oct). Our teacher and our students are scored on the same 300 answers, so all
numbers in the table are comparable. A reply the program cannot read counts as "marks nothing", which is what
an app would show.

In [ ]:
# Step 12: phone-sized chat models as the checker (about 30 minutes)
SELF = {'selfcheck05': 'qwen2.5-0.5b', 'selfcheck15': 'qwen2.5-1.5b'}
for name, model in SELF.items():
    assert run('llmlabel', '--name', name, '--local', model, '--on', 'test', '--limit', 300), 'Labelling failed.'
    free_disk()
if (WORK / 'llm' / 'qwen7b' / 'test').exists():   # the 7B labeller of 19 Oct; its replies are saved, so no GPU work
    run('llmlabel', '--name', 'qwen7b', '--local', 'qwen2.5-7b', '--on', 'test', '--limit', 300)

SIZE = {'selfcheck05': '0.5B', 'selfcheck15': '1.5B', 'qwen7b': '7B', 'ld_large': '396M', 'ld_base': '150M',
        'tiny17m': '17M', 'tiny68m': '68M', 'hard17m-8ep': '17M', 'kd17m': '17M', 'hard68m-8ep': '68M', 'kd68m': '68M'}
res = {n: json.loads((WORK / 'results' / f'llmlabel_{n}_test.json').read_text())
       for n in ['selfcheck05', 'selfcheck15', 'qwen7b'] if (WORK / 'results' / f'llmlabel_{n}_test.json').exists()}
ref = res['selfcheck15']['all_answers']
rows = ['| Checker | Size | Span F1 on the 300 answers [95%] | Minus Qwen2.5-1.5B [95%] | Replies it could not read |',
        '|---|---|---|---|---|']
for n, r in res.items():
    a = r['all_answers']['labeller']
    rows.append(f"| {r['llm_model']} (chat model, asked to list unsupported parts) | {SIZE[n]} | "
                f"{a['span_f1']} {a['ci95']} | | {r['unusable_replies']} of {r['answers']} |")
for n, c in ref['checkers'].items():
    rows.append(f"| {n} | {SIZE.get(n, '')} | {c['span_f1']} {c['ci95']} | "
                f"{c['span_f1'] - ref['labeller']['span_f1']:+.2f} {c['minus_labeller_ci95']} | |")
display(Markdown('\n'.join(rows)))

**How to read it.** If the 0.5B and 1.5B chat models score far below our 17M and 68M checkers (the last column
is positive with an interval above 0), the phone cannot simply ask its own model: a separate small checker is
needed, and that is the case for our work. Step 20 repeats this on our own test set. When it is done, tell
Claude "step 12 done".

## Step 13 — The 8-bit files and what they cost

The phone will run our checker as an 8-bit ONNX file. This step makes the 8-bit files of `kd68m` and `kd17m`
(the seed-42 runs) and scores them on **all** 2,700 RAGTruth test answers and the 4,000 PsiloQA answers, against
the same models in full precision. It also records the file sizes.

**It needs no GPU:** choose Runtime > Change runtime type > **CPU** (this saves your free GPU time), run the two
setup cells, then this cell. It takes about 2 hours, one answer at a time as on a phone. The ONNX files stay on
Colab's own disk, so Drive does not fill up; only small result files go to Drive. If the session ends, run the
setup cells and this cell again: finished parts are skipped.

In [ ]:
# Step 13: 8-bit files of kd68m and kd17m, their size and their accuracy (CPU only, about 2 hours)
ONNX = Path('/content/onnx')                          # Colab's own disk, not Drive
for name in ('kd68m', 'kd17m'):
    assert run('export', '--name', name, '--onnx-dir', ONNX), f'Export of {name} failed; see the message above.'
    assert run('check8bit', '--name', name, '--n', 2700, '--psilo', '--onnx-dir', ONNX), \
        f'The 8-bit check of {name} failed; see the message above.'

rows = ['| Checker | File size: full → 8-bit (MB) | RAGTruth: full → 8-bit, change [95%] | PsiloQA: full → 8-bit, change [95%] |',
        '|---|---|---|---|']
for name in ('kd68m', 'kd17m'):
    e = json.loads((WORK / 'results' / f'{name}_export.json').read_text())
    r = json.loads((WORK / 'results' / f'{name}_int8_n2700.json').read_text())
    q = json.loads((WORK / 'results' / f'{name}_int8_psilo.json').read_text())['groups']['all']
    rows.append(f"| {name} | {e['fp32_mb']} → {e['int8_mb']} | {r['span_f1_full_precision']} → {r['span_f1_int8']}, "
                f"{r['change']:+.2f} {r['change_ci95']} | {q['span_f1_full_precision']} → {q['span_f1_int8']}, "
                f"{q['change']:+.2f} {q['change_ci95']} |")
display(Markdown('\n'.join(rows)))

**How to read it.** Success criterion S5 needs the 8-bit `kd68m` to lose at most 1 point on RAGTruth (a change of
−1.00 or better) and its 8-bit file to be at most 150 MB. When it is done, tell Claude "step 13 done". Step 14
then times the 8-bit file on your laptop.

## Next steps

The cells for steps 9–11, 17 and 20 are added here when we reach them.